SMS SPAM DETECTION MODEL EVALUATION

In [ ]:
# install project in editable mode so local packages (if any) are available in the notebook environment
%pip install -e .

import os  


import pandas as pd
import numpy as np
import argparse
import joblib
import sys
import json
import ast
import mysql.connector

from tqdm import tqdm
from datetime import datetime

Load data

In [2]:
con = mysql.connector.connect(
    host = '10.168.51.196',
    port = 3306,
    user = 'unified',
    password = 'unified'
)

cur = con.cursor()

query = """
    select  	
        ml.id,
        data.payload,
        ml.embedding,
        ml.spam_label,
        ml.confidence_score
    from sms_spam_cd.ml_spam_result ml
    inner join sms_spam_cd.data_tdr_spam_filter data on data.id = ml.id
"""

cur.execute(query)

data = pd.DataFrame(cur.fetchall(), columns=['id', 'message', 'embedding', 'spam_label', 'confidence_score'])

data.head()

,id,message,embedding,spam_label,confidence_score
0,236791444,Chellum now too cool because all ready raining...,"b'[0.05234536528587341, -0.016886768862605095,...",0,0.93839
1,236791445,Assalamualaikum…\nKalo ada no wasap aku hang t...,"b'[0.0032701280433684587, -0.10649462044239044...",1,0.57340
2,236792627,Dont make fun,"b'[0.05241803452372551, -0.07558398693799973, ...",0,0.93631
3,236793544,12 dah,"b'[0.03576604276895523, -0.12311481684446335, ...",0,0.93165
4,236793760,168.5,"b'[0.029572995379567146, -0.1148369237780571, ...",0,0.92847


Preprocess 

In [3]:
# separate `embedding` column from `data`
embedding = data.pop('embedding')

# convert `embedding` to numpy.ndarray format
embeddings = [] 
for row in tqdm(embedding):
    embedding = row.decode('utf-8')  
    embeddings.append(ast.literal_eval(embedding)) 
    
embeddings = np.array(embeddings) 

100%|██████████| 31149/31149 [02:15<00:00, 229.89it/s]
